# HoopsAI 🏀🏀🏀

Welcome to the HoopsAI notebook!
This project focuses on building machine learning models to predict NBA game outcomes based on historical team statistics. We use regression to predict the point differential and classification to predict the win/loss outcome.

## Table of Contents

- HoopsAI: Predicting NBA Game Outcomes
- Setup
- Data Collection
- Feature Engineering
- Feature Construction
- Regression Task: Predicting Point Differential
- Classification Task: Predicting Win/Loss
- Hyperparameter Tuning
- Future Improvements
- Overall Structure


## Setup

Install Swar Patel's nba_api for nba.com
For more information visit: https://github.com/swar/nba_api?tab=readme-ov-file

We install the nba_api package to fetch NBA game data.

In [1]:
%pip install nba_api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.9/284.9 kB 4.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import nba_api
import numpy as np
from tqdm import tqdm
from nba_api.stats.endpoints import leaguegamelog
import time
import matplotlib.pyplot as plt
import seaborn as sns
import shap

## Data Collection

### Import Libraries
We import essential libraries like pandas, numpy, nba_api, matplotlib, seaborn, shap, and tqdm.

We download regular season game logs for every NBA season from 2000-2001 through 2024-2025.



In [3]:
all_seasons = [f"{year}-{str(year+1)[-2:]}" for year in range(2000, 2025)]
all_games = []

for season in tqdm(all_seasons):
    try:
        log = leaguegamelog.LeagueGameLog(
            season=season,
            season_type_all_star='Regular Season',
            player_or_team_abbreviation='T'
        )
        df_season = log.get_data_frames()[0]
        df_season['season'] = season
        all_games.append(df_season)
        time.sleep(1.2)
    except Exception as e:
        print(f"Failed season: {season}, error: {e}")

if all_games:
    games_df = pd.concat(all_games, ignore_index=True)
else:
    raise ValueError("No games data loaded")


100%|██████████| 25/25 [01:02<00:00,  2.51s/it]


### Combine and Clean Data
We standardize column names, filter important columns (e.g., points, field goals, turnovers), and sort by game date.

In [4]:
games_df.columns = [col.lower() for col in games_df.columns]
keep_cols = ['game_id', 'team_id', 'team_abbreviation', 'game_date', 'matchup', 'wl', 'pts', 'fgm', 'fga', 'fg3m',
             'fg3a', 'ftm', 'fta', 'oreb', 'dreb', 'ast', 'tov', 'stl', 'blk', 'pf', 'plus_minus', 'season']
games_df = games_df[keep_cols]
games_df['game_date'] = pd.to_datetime(games_df['game_date'])
games_df = games_df.sort_values(by='game_date')

## Feature Engineering

### Create Rolling & Cumulative Features
We generate 5-game and 10-game rolling averages for key stats like points, field goals, assists, and rebounds to better capture recent team performance.

In [5]:
def add_rolling_features(df):
    rolling_features = ['pts', 'fgm', 'fga', 'fg3m', 'fg3a', 'ftm', 'fta', 'oreb',
                        'dreb', 'ast', 'tov', 'stl', 'blk', 'pf', 'plus_minus']
    df = df.copy()
    for window in [5, 10]:
        for feat in rolling_features:
            rolling_col = f'rolling_{window}_{feat}'
            df[rolling_col] = (
                df.groupby('team_id')[feat]
                  .transform(lambda x: x.rolling(window, min_periods=1).mean().shift(1))
            )
    return df
games_df = add_rolling_features(games_df)

### Build Matchup-Level Dataset

We transform the team-level stats into game-level rows (Team A vs Team B) for model training.

In [6]:
def create_game_rows(df):
    grouped = df.groupby('game_id')
    rows = []

    for gid, group in grouped:
        if len(group) != 2:
            continue
        team_a, team_b = group.iloc[0], group.iloc[1]
        row = {
            'game_id': gid,
            'date': team_a['game_date'],
            'season': team_a['season'],
            'team_a': team_a['team_abbreviation'],
            'team_b': team_b['team_abbreviation'],
            'a_score': team_a['pts'],
            'b_score': team_b['pts']
        }
        for col in [col for col in df.columns if col.startswith('rolling')]:
            base = col
            row[f'a_{base}'] = team_a[col]
            row[f'b_{base}'] = team_b[col]
        rows.append(row)
    return pd.DataFrame(rows)

games_dataset = create_game_rows(games_df)
games_dataset = games_dataset.dropna().reset_index(drop=True)

print("games_dataset shape:", games_dataset.shape)
print(games_dataset.head())


✅ games_dataset shape: (30008, 67)
      game_id       date   season team_a team_b  a_score  b_score  \
0  0020000015 2000-11-01  2000-01    TOR    PHI       98      104   
1  0020000017 2000-11-01  2000-01    CHH    WAS       77       95   
2  0020000018 2000-11-01  2000-01    SAC    CLE      100      102   
3  0020000019 2000-11-01  2000-01    LAL    UTA       92       97   
4  0020000021 2000-11-02  2000-01    NYK    ATL       94       69   

   a_rolling_5_pts  b_rolling_5_pts  a_rolling_5_fgm  ...  a_rolling_10_tov  \
0             95.0            101.0             35.0  ...              15.0   
1            106.0             86.0             35.0  ...              17.0   
2            100.0             86.0             40.0  ...              18.0   
3             96.0            107.0             36.0  ...              20.0   
4             72.0             82.0             25.0  ...              22.0   

   b_rolling_10_tov  a_rolling_10_stl  b_rolling_10_stl  a_rolling_10_blk  

### Create Safe Feature Differences

We compute feature differences between teams for rolling stats that do not directly leak points (e.g., avoid using raw points scored).

In [7]:
good_metrics = [
    'fgm', 'fga', 'fg3m', 'fg3a', 'ftm', 'fta',
    'oreb', 'dreb', 'ast', 'tov', 'stl', 'blk', 'pf'
]


### Feature Construction

The create_feature_diffs_clean function builds feature differences between Team A and Team B using only a safe subset of rolling statistics that avoid leaking direct game outcomes.

For each metric, the function computes both:

* 5-game rolling difference

* 10-game rolling difference

These become the features (X) for model training.

In [8]:
good_metrics = ['fgm', 'fga', 'fg3m', 'fg3a', 'ftm', 'fta',
                'oreb', 'dreb', 'ast', 'tov', 'stl', 'blk', 'pf']

def create_feature_diffs_clean(df, metrics):
    feature_diffs = {}
    for metric in metrics:
        a_col = f'a_rolling_5_{metric}'
        b_col = f'b_rolling_5_{metric}'
        if a_col in df.columns and b_col in df.columns:
            feature_diffs[f'{metric}_5_diff'] = df[a_col] - df[b_col]

        a_col = f'a_rolling_10_{metric}'
        b_col = f'b_rolling_10_{metric}'
        if a_col in df.columns and b_col in df.columns:
            feature_diffs[f'{metric}_10_diff'] = df[a_col] - df[b_col]
    return pd.DataFrame(feature_diffs)

X = create_feature_diffs_clean(games_dataset, good_metrics)
y = games_dataset['a_score'] - games_dataset['b_score']

print("Safe X shape:", X.shape)
print(X.head())


✅ Safe X shape: (30008, 26)
   fgm_5_diff  fgm_10_diff  fga_5_diff  fga_10_diff  fg3m_5_diff  \
0        -3.0         -3.0        27.0         27.0          2.0   
1         2.0          2.0       -10.0        -10.0          1.0   
2         8.0          8.0        -5.0         -5.0          3.0   
3        -7.0         -7.0       -22.0        -22.0          0.0   
4        -5.0         -5.0       -11.0        -11.0         -3.0   

   fg3m_10_diff  fg3a_5_diff  fg3a_10_diff  ftm_5_diff  ftm_10_diff  ...  \
0           2.0          9.0           9.0        -2.0         -2.0  ...   
1           1.0          2.0           2.0        15.0         15.0  ...   
2           3.0          6.0           6.0        -5.0         -5.0  ...   
3           0.0          1.0           1.0         3.0          3.0  ...   
4          -3.0         -4.0          -4.0         3.0          3.0  ...   

   ast_5_diff  ast_10_diff  tov_5_diff  tov_10_diff  stl_5_diff  stl_10_diff  \
0         0.0          0.0

## Regression Task: Predicting Point Differential

### Prepare Data
We split the feature matrix X and target y into training and test sets using an 80/20 split.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### Baseline Models

We fit a Dummy Regressor that predicts the mean point differential to establish a simple baseline for comparison.

In [10]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
print("Baseline MAE:", mean_absolute_error(y_test, y_pred_dummy))

Baseline MAE: 11.201643621639299


### Model Training

We train and evaluate three machine learning models:

* Ridge Regression (linear model)

* Random Forest Regressor (ensemble tree model)

* XGBoost Regressor (gradient boosting model)

Each model's performance is reported using:

* Mean Absolute Error (MAE)

* Root Mean Squared Error (RMSE)

* R² Score (explained variance)


In [11]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Ridge": Ridge(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, objective='reg:squarederror', random_state=42)
}

def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
    print(f"R²: {r2_score(y_test, y_pred):.4f}")

for name, model in models.items():
    print(f"\n{name}")
    evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test)



Ridge
MAE: 10.3808
RMSE: 13.1479
R²: 0.1002

RandomForest
MAE: 10.5530
RMSE: 13.3371
R²: 0.0741

XGBoost
MAE: 10.8118
RMSE: 13.6823
R²: 0.0256


That Wasn't good... Blah blah blah sports books blah blah betting blah they are bad at it too.

## Classification Task: Predicting Win/Loss

### Create Win/Loss Labels
We convert the point differential target into a binary classification task:

* 1: Team A wins

* 0: Team A loses

In [18]:

y_class = (y > 0).astype(int)

print("y_class distribution:\n", y_class.value_counts())

y_class distribution:
 0    15121
1    14887
Name: count, dtype: int64


### Prepare Classification Data
We reuse the same features X, but now our target variable is the win/loss label (y_class).

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_class, test_size=0.2, random_state=42)

scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)


### Train Classifiers
We train and evaluate three classification models:

* Logistic Regression

* Random Forest Classifier

* XGBoost Classifier

Each classifier is evaluated using:

* Accuracy

* F1 Score

* OC AUC (if available)

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost Classifier": XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss', random_state=42)
}

def evaluate_classifier(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")

for name, model in classifiers.items():
    print(f"\n{name}")
    evaluate_classifier(model, X_train_c_scaled, y_train_c, X_test_c_scaled, y_test_c)



Logistic Regression
Accuracy: 0.6155
F1 Score: 0.6089
ROC AUC: 0.6671

Random Forest Classifier
Accuracy: 0.5968
F1 Score: 0.5839
ROC AUC: 0.6307

XGBoost Classifier


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [01:01:52] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 0.5870
F1 Score: 0.5833
ROC AUC: 0.6246


Not bad but not great let's do some tuning.

## Hyperparameter Tuning

Grid Search for Best Parameters
We perform GridSearchCV to fine-tune hyperparameters for:

* Logistic Regression: Regularization strength (C)

* Random Forest: Number of trees, max depth, min samples split

* XGBoost: Number of trees, max depth, learning rate

The best parameters and cross-validation scores are printed for each model.

In [19]:
from sklearn.model_selection import GridSearchCV

log_reg = LogisticRegression(max_iter=1000)
param_grid_log = {'C': [0.01, 0.1, 1, 10, 100]}
grid_log = GridSearchCV(log_reg, param_grid_log, cv=3, scoring='accuracy')
grid_log.fit(X_train_c_scaled, y_train_c)

print("\nBest Logistic Regression:")
print(f"Best Params: {grid_log.best_params_}")
print(f"Best Score (CV Accuracy): {grid_log.best_score_:.4f}")

rf = RandomForestClassifier(random_state=42)
param_grid_rf = {'n_estimators': [100, 200],
                 'max_depth': [5, 10, 20],
                 'min_samples_split': [2, 5]}
grid_rf = GridSearchCV(rf, param_grid_rf, cv=3, scoring='accuracy')
grid_rf.fit(X_train_c_scaled, y_train_c)

print("\nBest Random Forest:")
print(f"Best Params: {grid_rf.best_params_}")
print(f"Best Score (CV Accuracy): {grid_rf.best_score_:.4f}")

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
param_grid_xgb = {'n_estimators': [100, 200],
                  'max_depth': [3, 5, 7],
                  'learning_rate': [0.01, 0.1, 0.2]}
grid_xgb = GridSearchCV(xgb, param_grid_xgb, cv=3, scoring='accuracy')
grid_xgb.fit(X_train_c_scaled, y_train_c)

print("\nBest XGBoost:")
print(f"Best Params: {grid_xgb.best_params_}")
print(f"Best Score (CV Accuracy): {grid_xgb.best_score_:.4f}")



Best Logistic Regression:
Best Params: {'C': 0.1}
Best Score (CV Accuracy): 0.6213

Best Random Forest:
Best Params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
Best Score (CV Accuracy): 0.6042


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [01:29:57] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [01:29:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [01:29:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [01:29:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [01:29:59] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_e


Best XGBoost:
Best Params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Best Score (CV Accuracy): 0.6093


## Future Improvements

* Add player-level data (e.g., injuries, roster depth).

* Incorporate Vegas betting lines to enhance predictions.

* Engineer synergy-based features (e.g., player combos).

* Build a web app to deploy live predictions.

* Explore deep learning models or graph-based models (GNNs).

## Overall Structure

| Section                          | Purpose                                              |
|:---------------------------------|:-----------------------------------------------------|
| Setup & Installations            | Install nba_api and import libraries                 |
| Data Collection                  | Download NBA game logs (2000-2025)                   |
| Feature Engineering              | Rolling averages, matchup building, feature diffs    |
| Feature Construction             | Create feature differences safely                   |
| Regression Modeling              | Predict point differential                          |
| Classification Modeling          | Predict win/loss outcome                             |
| Hyperparameter Tuning            | Optimize model performance                          |
| Future Directions                | Discuss enhancements and deployments                 |
